In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Data Engineering\Projects\fleetpulse_project")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print(f"Project root added:{PROJECT_ROOT}")

Project root added:C:\Data Engineering\Projects\fleetpulse_project


In [2]:
from scripts.utils.spark_session import create_spark_session
from scripts.utils.config import (
    BRONZE_BASE_PATH,
    SILVER_BASE_PATH
)

In [3]:
table = "customers"
spark = create_spark_session("DataFrameAPIPractise")
path =  f"{BRONZE_BASE_PATH}/{table}"

In [4]:
customers_df = spark.read.parquet(path)
customers_df.show(truncate=False)

+-----------+---------------+--------------------------+-------------+-------------------+----------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+--------------------------+--------------------------+----------+--------------------------+--------------+------------+----------------+
|customer_id|customer_number|customer_name             |customer_type|industry           |contact_email                     |contact_phone|billing_city|billing_state|billing_country|credit_status|customer_status|source_system|created_at                |updated_at                |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+-----------+---------------+--------------------------+-------------+-------------------+----------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+--------------------------+-------------------

In [5]:
import pyspark.sql.functions as F

customer_filter_df = customers_df.groupBy(F.col("customer_type")).agg(
    F.count(F.col("customer_id")).alias("customer_count"),
    F.sum(F.when(F.col("customer_status")== "ACTIVE",1).otherwise(0)).alias("active_customers"),
    F.sum(F.when(F.col("customer_status")== "INACTIVE",1).otherwise(0)).alias("inactive_customers"),
    F.sum(F.when(F.col("credit_status")== "GOOD",1).otherwise(0)).alias("good_credit_customers"),
    F.sum(F.when(F.col("credit_status")== "HOLD",1).otherwise(0)).alias("hold_credit_customers"),
    F.sum(F.when(F.col("credit_status")== "REVIEW",1).otherwise(0)).alias("review_credit_customers")
)

customer_filter_df.show(truncate=False)

+-------------+--------------+----------------+------------------+---------------------+---------------------+-----------------------+
|customer_type|customer_count|active_customers|inactive_customers|good_credit_customers|hold_credit_customers|review_credit_customers|
+-------------+--------------+----------------+------------------+---------------------+---------------------+-----------------------+
|GOVERNMENT   |3             |3               |0                 |2                    |0                    |1                      |
|COMMERCIAL   |12            |11              |1                 |6                    |3                    |3                      |
+-------------+--------------+----------------+------------------+---------------------+---------------------+-----------------------+



In [6]:
table = "branches"
path = f"{BRONZE_BASE_PATH}/{table}"
branches_df =  spark.read.parquet(path)
branches_df.show(truncate=False)

+---------+-----------+------------------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|branch_id|branch_code|branch_name                   |region   |district     |city      |state|country|postal_code|latitude |longitude |branch_status|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+---------+-----------+------------------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|1        |BR-OH-001  |Youngstown Rental Center      |Midwest  |Ohio Valley  |Youngstown|OH   |USA  

In [7]:
filtered_branches_df =  branches_df.filter(F.col("is_deleted") == "false")


trimmed_branches_df = filtered_branches_df.select(
    "branch_id",
    "region",
    "state",
    "branch_status"
)

trimmed_branches_df.show(truncate=False)


branch_metrics_df =  trimmed_branches_df.groupBy("region", "state").agg(
    F.count(F.col("branch_id")).alias("total_branches"),
    F.sum(F.when(F.upper(F.trim(F.col("branch_status"))) == "ACTIVE", 1 ).otherwise(0)).alias("active_branches"),
    F.sum(F.when(F.upper(F.trim(F.col("branch_status"))) != "ACTIVE", 1 ).otherwise(0)).alias("inactive_branches"),
)

branch_metrics_df.show(truncate=False)
branch_metrics_df.orderBy(
    F.col("region").asc(),
    F.col("state").asc()
    ).show(truncate=False)

+---------+---------+-----+-------------+
|branch_id|region   |state|branch_status|
+---------+---------+-----+-------------+
|1        |Midwest  |OH   |ACTIVE       |
|2        |Midwest  |OH   |ACTIVE       |
|3        |Northeast|PA   |ACTIVE       |
|4        |Midwest  |MI   |ACTIVE       |
+---------+---------+-----+-------------+

+---------+-----+--------------+---------------+-----------------+
|region   |state|total_branches|active_branches|inactive_branches|
+---------+-----+--------------+---------------+-----------------+
|Midwest  |OH   |2             |2              |0                |
|Northeast|PA   |1             |1              |0                |
|Midwest  |MI   |1             |1              |0                |
+---------+-----+--------------+---------------+-----------------+

+---------+-----+--------------+---------------+-----------------+
|region   |state|total_branches|active_branches|inactive_branches|
+---------+-----+--------------+---------------+-----------

In [8]:
small_customer_df = customers_df.select(
    "customer_id",
    "customer_type",
    "credit_status"
)

small_customer_df.explain(mode="formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [customer_id#0L, customer_type#3, credit_status#10]
Batched: true
Location: InMemoryFileIndex [s3a://fleetpulse-lake/bronze/rental_oltp/customers]
ReadSchema: struct<customer_id:bigint,customer_type:string,credit_status:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [customer_id#0L, customer_type#3, credit_status#10]




In [9]:
active_customers_df = customers_df.filter(
    F.col("customer_status") == "ACTIVE"
).select(
    "customer_id",
    "customer_type",
    "customer_status"
)

active_customers_df.explain(mode="formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [customer_id#0L, customer_type#3, customer_status#11]
Batched: true
Location: InMemoryFileIndex [s3a://fleetpulse-lake/bronze/rental_oltp/customers]
PushedFilters: [IsNotNull(customer_status), EqualTo(customer_status,ACTIVE)]
ReadSchema: struct<customer_id:bigint,customer_type:string,customer_status:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [customer_id#0L, customer_type#3, customer_status#11]

(3) Filter [codegen id : 1]
Input [3]: [customer_id#0L, customer_type#3, customer_status#11]
Condition : (isnotnull(customer_status#11) AND (customer_status#11 = ACTIVE))




In [10]:
partitions = customers_df.rdd.getNumPartitions()
print(partitions)

1


In [11]:
customer_summary_df = customers_df.groupBy("customer_type").agg(
    F.count("*").alias("total_customers")
)

customer_summary_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [customer_type#3]
Batched: true
Location: InMemoryFileIndex [s3a://fleetpulse-lake/bronze/rental_oltp/customers]
ReadSchema: struct<customer_type:string>

(2) HashAggregate
Input [1]: [customer_type#3]
Keys [1]: [customer_type#3]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#336L]
Results [2]: [customer_type#3, count#337L]

(3) Exchange
Input [2]: [customer_type#3, count#337L]
Arguments: hashpartitioning(customer_type#3, 200), ENSURE_REQUIREMENTS, [plan_id=274]

(4) HashAggregate
Input [2]: [customer_type#3, count#337L]
Keys [1]: [customer_type#3]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#335L]
Results [2]: [customer_type#3, count(1)#335L AS total_customers#314L]

(5) AdaptiveSparkPlan
Output [2]: [customer_type#3, total_customers#314L]
Arguments: isFinalPlan=false




In [12]:
from pyspark.sql.window import Window
table = "equipment_status_history"
path = f"{BRONZE_BASE_PATH}/{table}"
equipment_status_history_df = spark.read.parquet(path)

window_spec = Window.partitionBy("equipment_id").orderBy(F.col("status_start_timestamp").desc())

df_ranked = equipment_status_history_df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)

df_ranked.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- Window (4)
   +- Sort (3)
      +- Exchange (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [16]: [status_history_id#338L, equipment_id#339L, branch_id#340L, old_status#341, new_status#342, status_change_reason#343, status_start_timestamp#344, status_end_timestamp#345, source_system#346, created_at#347, updated_at#348, is_deleted#349, ingestion_timestamp#350, ingestion_date#351, source_table#352, bronze_load_type#353]
Batched: true
Location: InMemoryFileIndex [s3a://fleetpulse-lake/bronze/rental_oltp/equipment_status_history]
ReadSchema: struct<status_history_id:bigint,equipment_id:bigint,branch_id:bigint,old_status:string,new_status:string,status_change_reason:string,status_start_timestamp:timestamp,status_end_timestamp:timestamp,source_system:string,created_at:timestamp,updated_at:timestamp,is_deleted:boolean,ingestion_timestamp:timestamp,ingestion_date:date,source_table:string,bronze_load_type:string>

(2) Exchange
Input [16]

In [13]:
df1 = customers_df.select("customer_id", "customer_type")
df2 = df1.filter(F.col("customer_type") == "COMMERCIAL")
df3 = df2.groupBy("customer_type").count()
df3.explain(mode="extended")

== Parsed Logical Plan ==
'Aggregate ['customer_type], ['customer_type, 'count(1) AS count#357]
+- Filter (customer_type#3 = COMMERCIAL)
   +- Project [customer_id#0L, customer_type#3]
      +- Relation [customer_id#0L,customer_number#1,customer_name#2,customer_type#3,industry#4,contact_email#5,contact_phone#6,billing_city#7,billing_state#8,billing_country#9,credit_status#10,customer_status#11,source_system#12,created_at#13,updated_at#14,is_deleted#15,ingestion_timestamp#16,ingestion_date#17,source_table#18,bronze_load_type#19] parquet

== Analyzed Logical Plan ==
customer_type: string, count: bigint
Aggregate [customer_type#3], [customer_type#3, count(1) AS count#357L]
+- Filter (customer_type#3 = COMMERCIAL)
   +- Project [customer_id#0L, customer_type#3]
      +- Relation [customer_id#0L,customer_number#1,customer_name#2,customer_type#3,industry#4,contact_email#5,contact_phone#6,billing_city#7,billing_state#8,billing_country#9,credit_status#10,customer_status#11,source_system#12,cre

In [14]:
customers_df.show(truncate=False)

+-----------+---------------+--------------------------+-------------+-------------------+----------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+--------------------------+--------------------------+----------+--------------------------+--------------+------------+----------------+
|customer_id|customer_number|customer_name             |customer_type|industry           |contact_email                     |contact_phone|billing_city|billing_state|billing_country|credit_status|customer_status|source_system|created_at                |updated_at                |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+-----------+---------------+--------------------------+-------------+-------------------+----------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+--------------------------+-------------------

In [15]:
filtered_customer_df = customers_df.groupBy(F.col("customer_type")).agg(
    F.count(F.col("customer_id")).alias("total_customers"),
    F.sum(F.when(F.upper(F.trim(F.col("customer_status"))) == "ACTIVE", 1).otherwise(0)).alias("active_customers"),
    F.sum(F.when(F.col("customer_status")== "INACTIVE",1).otherwise(0)).alias("inactive_customers"),
    F.sum(F.when(F.col("credit_status")== "GOOD",1).otherwise(0)).alias("good_credit_customers"),
    F.sum(F.when(F.col("credit_status")== "HOLD",1).otherwise(0)).alias("hold_credit_customers"),
    F.sum(F.when(F.col("credit_status")== "REVIEW",1).otherwise(0)).alias("review_credit_customers")
)

filtered_customer_df.explain(mode="formatted")
filtered_customer_df.show(truncate=False)

customers_df.rdd.getNumPartitions()
filtered_customer_df.rdd.getNumPartitions()

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [customer_id#0L, customer_type#3, credit_status#10, customer_status#11]
Batched: true
Location: InMemoryFileIndex [s3a://fleetpulse-lake/bronze/rental_oltp/customers]
ReadSchema: struct<customer_id:bigint,customer_type:string,credit_status:string,customer_status:string>

(2) HashAggregate
Input [4]: [customer_id#0L, customer_type#3, credit_status#10, customer_status#11]
Keys [1]: [customer_type#3]
Functions [6]: [partial_count(customer_id#0L), partial_sum(CASE WHEN (upper(trim(customer_status#11, None)) = ACTIVE) THEN 1 ELSE 0 END), partial_sum(CASE WHEN (customer_status#11 = INACTIVE) THEN 1 ELSE 0 END), partial_sum(CASE WHEN (credit_status#10 = GOOD) THEN 1 ELSE 0 END), partial_sum(CASE WHEN (credit_status#10 = HOLD) THEN 1 ELSE 0 END), partial_sum(CASE WHEN (credit_status#10 = REVIEW) THEN 1 ELSE 0 END)]
Aggregate 

1

In [16]:
equipment_status_history_df.show(truncate=False)

+-----------------+------------+---------+-----------+-----------+-------------------------------------------+----------------------+--------------------+-------------+--------------------------+--------------------------+----------+--------------------------+--------------+------------------------+----------------+
|status_history_id|equipment_id|branch_id|old_status |new_status |status_change_reason                       |status_start_timestamp|status_end_timestamp|source_system|created_at                |updated_at                |is_deleted|ingestion_timestamp       |ingestion_date|source_table            |bronze_load_type|
+-----------------+------------+---------+-----------+-----------+-------------------------------------------+----------------------+--------------------+-------------+--------------------------+--------------------------+----------+--------------------------+--------------+------------------------+----------------+
|1                |1           |1        |AVAI

In [17]:
window_spec_1 = Window.partitionBy(F.col("equipment_id")).orderBy(F.col("status_start_timestamp").desc())

rowed_equipment_status_history_df = equipment_status_history_df.withColumn(
    "row_number", F.row_number().over(window_spec_1)
)
filtered_equipment_status_history_df = rowed_equipment_status_history_df.filter(F.col("row_number") == 1 ).orderBy(F.col("equipment_id").asc())

filtered_equipment_status_history_df.show(truncate=False)
filtered_equipment_status_history_df.explain(mode="formatted")



+-----------------+------------+---------+-----------+-----------+-----------------------------------+----------------------+--------------------+-------------+--------------------------+--------------------------+----------+--------------------------+--------------+------------------------+----------------+----------+
|status_history_id|equipment_id|branch_id|old_status |new_status |status_change_reason               |status_start_timestamp|status_end_timestamp|source_system|created_at                |updated_at                |is_deleted|ingestion_timestamp       |ingestion_date|source_table            |bronze_load_type|row_number|
+-----------------+------------+---------+-----------+-----------+-----------------------------------+----------------------+--------------------+-------------+--------------------------+--------------------------+----------+--------------------------+--------------+------------------------+----------------+----------+
|8                |1           |1    

In [19]:
table = "rental_contracts"
path = f"{BRONZE_BASE_PATH}/{table}"
rental_contracts_df = spark.read.parquet(path)
rental_contracts_df.show(truncate=False)
customers_df.show(truncate=False)

+-----------+---------------+-----------+---------+---------------------+-----------------+--------------------+------------------+---------------+-------------+-------------+---------------+---------------+----------+------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+----------------+----------------+
|contract_id|contract_number|customer_id|branch_id|contract_created_date|rental_start_date|expected_return_date|actual_return_date|contract_status|sales_channel|payment_terms|subtotal_amount|discount_amount|tax_amount|total_amount|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table    |bronze_load_type|
+-----------+---------------+-----------+---------+---------------------+-----------------+--------------------+------------------+---------------+-------------+-------------+---------------+---------------+----------+------------